# Recombining Trees: Conceptual Overview
Let's explore a different type of tree structure: the recombining tree. This type of tree has __fewer unique nodes__ compared to traditional trees, as it allows for shared paths and recombination of states.

To understand this concept better, we'll first examine the key characteristics of recombining trees and how they differ from conventional tree structures. Then we'll do an example of a classical application of recombining trees in the context of pricing an asset, e.g., shares of stock, a commodity, or any other financial instrument.

Let's get started!
___

<div>
    <center>
      <img
        src="figs/Fig-BinaryTree-w-Replacement.svg"
        alt="Binary Tree with Recombination"
        height="500"
        width="700"
      />
    </center>
  </div>

__Figure__: in a recombining tree, different paths can lead to the same state. Notice how the "up-then-down" and "down-then-up" paths both arrive at the same middle node at level 2. By including only unique paths, we dramatically reduce memory requirements compared to a full binary tree, which would have $2^h$ leaf nodes, while a recombining tree has only $h+1$ unique states at each level.

## Motivation: Why Recombining Trees Matter

Recombining trees solve a fundamental computational problem in financial modeling and stochastic processes. A traditional n-ary tree grows exponentially, i.e., we have $n^h$ nodes at depth $h$, making deep trees computationally intractable. For example, a 20-step binary tree would require over one million leaf nodes.

> __Recombining trees exploit state equivalence:__ if two paths lead to the same underlying state (same price, same inventory level, etc.), we can merge them into a single node.  This reduces storage from exponential $O(n^h)$ to polynomial $O(h^n)$, enabling practical computation of multi-step processes. 

The mathematical elegance lies in recognizing that only the cumulative effects matter, not the specific sequence of moves. For example, a stock that goes up twice and down once reaches the same price as an up, down, up sequence. Thus, we can represent all such paths with a single node.

## Theory

A recombining n-ary tree models $n$ possible outcomes per step but merges paths that arrive at the same state. If you prefer a code-first view, you can skip ahead to the example and return here after a first run; the formulas below explain the node counts and indexing strategy that the code relies on.

After $k$ steps the state is captured by a nonnegative count vector $\mathbf{c} \in \mathbb{N}_0^{n}$ with total $k$ (the number of times each outcome in $\Delta$ was taken), so all path permutations that yield the same $\mathbf{c}$ share one node. The number of unique nodes at level $k$ is given by the stars-and-bars count
$$
L_k = \binom{k + n - 1}{n - 1}.
$$
Summing over levels $k = 0,\dots,h$ gives the total number of unique nodes in a height-$h$ tree:
$$
T(h, n) = \sum_{k=0}^{h} L_k = \binom{h + n}{n} = \binom{h + n}{h}.
$$
When storing nodes in level order in a flat array, the offset to the first node of level $\ell$ is the number of nodes in all prior levels:
$$
O(\ell) = \sum_{k=0}^{\ell - 1} L_k = \binom{\ell + n - 1}{n}.
$$

A node’s global flat index for count vector $\mathbf{c}$ at level $\ell$ is then
$$
\operatorname{index}(\mathbf{c}, \ell) = O(\ell) + \operatorname{rank}_{\ell}(\mathbf{c}),
$$
where $\operatorname{rank}_{\ell}(\mathbf{c})$ is its zero-based rank within level $\ell$ under a fixed ordering of count vectors. A convenient within-level ordering is lexicographic on $\mathbf{c}$. Its rank admits a combinatorial recursion by sweeping the first coordinate and counting how many compositions remain:
$$
\operatorname{rank}(c_1,\ldots,c_n; k)
= \sum_{a=0}^{c_1-1} \binom{k - a + n - 2}{n - 2}
\; + \; \operatorname{rank}(c_2,\ldots,c_n; k - c_1),
$$
with the base case $\operatorname{rank}(c_n; \cdot) = 0$. This yields a stable, deterministic index for any $\mathbf{c}$ with $\sum_j c_j = k$. As a sanity check, the binary case $n = 2$ yields
$$
L_k = k + 1, \quad
T(h,2) = \frac{(h + 1)(h + 2)}{2}, \quad
O(\ell) = \binom{\ell + 1}{2} = \frac{\ell(\ell + 1)}{2}.
$$
These match the familiar triangular numbers and the usual recombining binomial tree layout. In applications like pricing, any multiplicative state quantity will factor through the counts, e.g.
$$
\text{price} = \text{price}_0 \, \prod_{j=1}^{n} \Delta_j^{\,c_j},
$$
where $c_j$ is the count of times outcome $j$ was taken in the path to the current node, and $\Delta_j$ is the corresponding price change factor for outcome $j$. This is why collapsing permutations into a single node preserves the state.

___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

The [include command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

In [18]:
include("Include.jl");
using Test;

In addition to standard Julia libraries, we'll also use the `VLDataScienceMachineLearningPackage.jl` package. See the documentation for details on the functions, types, and data used in this material.

## Example: Binary Commodity Price Tree
In this example, we will build a binary price tree, i.e., a tree that assumes that the price of a good will go up or down in the next time period, e.g., tomorrow. To start, we'll set up some constants that we'll use, then we'll create a tree model, and then populate the data in the tree.

See the comment next to each constant for what it is, its permissible values, units, etc.

In [35]:
h = 2; # height of the tree {0,1,2} levels
n = 2; # branching factor of the tree, binary = 2
price = 100.0; # initial price of the good
Δt = (1/365); # time step (1 day assuming 365 days per year)
u = exp(0.1*Δt); # up-factor (multiplicative)
d = exp(-0.1*Δt); # down-factor (multiplicative)
Δ = [u, d]; # possible price changes, u or d

Ok, now we construct the recombining price tree using the constants we defined. We'll do this in two steps, a __build__ and __populate__ approach:

* __build__: We first build the tree model with the specified height and branching factor, which creates the connectivity information; then we populate the tree with the price and path data. To build the tree model, we [use a `build(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/factory/#VLDataScienceMachineLearningPackage.build) and pass in the height h and branching factor n (as a NamedTuple).
* __populate__: We then pipe the resulting model [into the `populate!(...)` method](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/factory/#VLDataScienceMachineLearningPackage.populate!) which updates the `data::Dict{Int64, NamedTuple}` field with price and path information. Notice that we use [the pipe `|>` operator](https://docs.julialang.org/en/v1/manual/functions/#Function-composition-and-piping) to pass the unpopulated model to [the `populate!(...)` method](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/factory/#VLDataScienceMachineLearningPackage.populate!).

If you prefer code-first, you can skip ahead to the next cell and circle back to this explanation after running it once.

In [31]:
my_tree_model = build(MyAdjacencyRecombiningCommodityPriceTree, (
    h = h, # what is the height of the tree
    n = n, # what is the branching factor
)) |> m -> populate!(m, price, Δ); # wow! Fancy ...

The `my_tree_model::MyAdjacencyRecombiningCommodityPriceTree` variable now holds a tree model which is configured with the specified height and branching factor, and contains the price and path data for the tree.

Let's check out some of the fields in the tree model, starting with the `connectivity::Dict{Int64, Vector{Int64}}` field.

In [32]:
my_tree_model.connectivity

Dict{Int64, Vector{Int64}} with 10 entries:
  0 => [1, 2]
  4 => [7, 8]
  5 => [8, 9]
  6 => [10, 11]
  2 => [4, 5]
  7 => [11, 12]
  9 => [13, 14]
  8 => [12, 13]
  3 => [6, 7]
  1 => [3, 4]

The `connectivity::Dict{Int64, Vector{Int64}}` maps each parent node’s index to the indices of its children at the next level. Keys are node indices (the root is `0`). Values are ordered vectors whose elements are the child indices of that node. Note: these node indices are zero-based (root `0`), which differs from Julia’s usual one-based array indexing.

> __Recombination:__ Recombination shows up when different parents point to the same child index—distinct paths that arrive at the same state share a node. Sketch (indices only):
> ```
> 0 → [1, 2]
> 1 → [3, 4]
> 2 → [4, 5]
> ```
> Here node `4` is a child of both `1` and `2`, illustrating recombination. 

Leaves (last level) don’t appear as keys in the connectivity map. Indices are compact integer labels assigned by the builder.

Next, let's look at the `data::Dict{Int64, NamedTuple}` field that stores the actual data for each node.

In [33]:
my_tree_model.data

Dict{Int64, NamedTuple} with 10 entries:
  0 => (price = 100.0, path = [0, 0])
  4 => (price = 100.0, path = [1, 1])
  5 => (price = 99.9452, path = [0, 2])
  6 => (price = 100.082, path = [3, 0])
  2 => (price = 99.9726, path = [0, 1])
  7 => (price = 100.027, path = [2, 1])
  9 => (price = 99.9178, path = [0, 3])
  8 => (price = 99.9726, path = [1, 2])
  3 => (price = 100.055, path = [2, 0])
  1 => (price = 100.027, path = [1, 0])

The `data::Dict{Int64, NamedTuple}` dictionary stores the per-node payload. The keys are the node indices (the same labels used in `connectivity` dictionary), and values are compact `NamedTuple`s with fields produced by the `populate!(...)` method. 

__Payload__: Each payload tuple contains a `price::Float64` and a `path::Vector{Int64}` value. The path counts moves in the same order as the $\Delta$-perturbation vector; for the binary case with $\Delta = [u, d]$, we have $\text{path} = [u_{\text{count}}, d_{\text{count}}]$, where $u_{\text{count}}$ is the count of up moves and $d_{\text{count}}$ is the count of down moves for each node.

__Price__: The price is computed multiplicatively as $\text{price} = \text{price}_{0}\times(u^{\text{path}[1]}) \times (d^{\text{path}[2]})$. Because the tree recombines, nodes with the same path (the same state) appear only once even if multiple parents point to them. Note that in this model the node indices are zero-based (the root is `0`), which differs from Julia’s usual one-based array indexing. Thus, we'll use the dictionary hack (or [the `OffsetArrays.jl` package](https://github.com/JuliaArrays/OffsetArrays.jl/tree/master)) to manage this.

Example output:
```
  0 => (price = 100.0,    path = [0, 0])   # root node (today)
  1 => (price = 100.027,  path = [1, 0])   # one up move
  2 => (price = 99.9726,  path = [0, 1])   # one down move
  3 => (price = 100.055,  path = [2, 0])   # two up moves
  4 => (price = 100.0,    path = [1, 1])   # up + down (recombines)
  5 => (price = 99.9452,  path = [0, 2])   # two down moves
```
These numbers are illustrative; your exact values depend on the constants above. You can access a node’s price via `my_tree_model.data[i].price` and derive its level with `sum(my_tree_model.data[i].path)`.

### Level from path
We don't need to store the tree `level::Int64` in the payload to use it; we can compute it on the fly as `level = sum(path)`. With the `path = [u_count, d_count]` data, the time step is `u_count + d_count`. This keeps the payload minimal.

In [23]:
# Helper: compute level for a node index without mutating payload
level_of(idx) = sum(my_tree_model.data[idx].path)

# Tiny smoke test: show index, path, and derived level for a few nodes
for idx in sort(collect(keys(my_tree_model.data)))[1:3]
    nt = my_tree_model.data[idx]
    println((idx = idx, path = nt.path, level = level_of(idx), price = nt.price))
end

(idx = 0, path = [0, 0], level = 0, price = 100.0)
(idx = 1, path = [1, 0], level = 1, price = 100.0274010136661)
(idx = 2, path = [0, 1], level = 1, price = 99.97260649243266)


### Tests: Mathematical Consistency

Let's verify that our tree output matches the theoretical pricing formula. For any node with path counts $[u_{count}, d_{count}]$, the price should equal $\text{price}_0 \cdot u^{u_{count}} \cdot d^{d_{count}}$.

**What's happening in our test suite:**

We're using Julia's `Test` module with **nested test sets**, which is a powerful way to organize related tests into logical groups. This is the first time we've used nested test structures in this course, so let's understand how it works.

The outer test set `@testset "Recombining Tree Verification"` groups all our tree-related tests together. Inside this, we have two inner test sets that focus on specific aspects: `"Pricing Formula Verification"` tests mathematical correctness, while `"Tree Structure Verification"` tests data integrity. This nested approach provides organized output and lets you run specific test groups independently.

> **Key testing concepts we're using:** The basic `@test condition` macro asserts that a condition must be true; if it evaluates to false, the test fails and reports exactly what went wrong. For floating-point comparisons, we use `@test a ≈ b atol = 1e-10` which tests approximate equality within a specified tolerance (floating-point arithmetic can introduce tiny rounding errors that would make exact equality unreliable).
>
> **What each test validates:** Our pricing tests loop through every node in the tree and calculate what the price should be using the theoretical formula $\text{price}_0 \cdot u^{u_{count}} \cdot d^{d_{count}}$, then verify this matches the actual stored price. The structure tests ensure data integrity by verifying that path counts are non-negative integers, that the root node correctly starts with path `[0,0]`, and that computed tree levels fall within expected bounds (non-negative and not exceeding the tree height).

The test suite provides both automated verification with clear pass/fail status and human-readable output showing the actual versus expected values for each node, making it easy to debug any issues that might arise.

In [ ]:
# NESTED TEST SETS: Organize related tests into logical groups
@testset "Recombining Tree Verification" begin
    
    # INNER TEST SET 1: Check mathematical correctness of pricing
    @testset "Pricing Formula Verification" begin
        println("Verifying pricing formula: price = price₀ * u^(u_count) * d^(d_count)")
        println("=" ^ 65)
        
        # Loop through all nodes in sorted order
        for idx in sort(collect(keys(my_tree_model.data)))
            nt = my_tree_model.data[idx]  # Get the node data
            
            # Calculate what the price SHOULD be using the theoretical formula
            expected_price = price * (u^nt.path[1]) * (d^nt.path[2])
            
            # TEST: Does actual price match expected? (≈ handles floating-point precision)
            @test nt.price ≈ expected_price atol=1e-10
            
            # Human-readable output showing the comparison
            println("Node $idx: path=$(nt.path), price=$(round(nt.price, digits=6)), expected=$(round(expected_price, digits=6)), ✓")
        end
        
        println("\nAll pricing formula tests passed! ✓")
    end
    
    # INNER TEST SET 2: Check data structure integrity
    @testset "Tree Structure Verification" begin
        
        # TEST: All path counts should be non-negative integers
        for (idx, nt) in my_tree_model.data
            @test all(nt.path .>= 0)  # all() returns true if every element satisfies condition
        end
        
        # TEST: Root node should start with no moves taken
        @test my_tree_model.data[0].path == [0, 0]
        
        # TEST: Level calculations should be sensible
        for (idx, nt) in my_tree_model.data
            computed_level = sum(nt.path)  # level = total number of moves
            @test computed_level >= 0      # can't have negative levels
            @test computed_level <= h      # can't exceed tree height
        end
        
        println("All structure tests passed! ✓")
    end
end;

Verifying pricing formula: price = price₀ * u^(u_count) * d^(d_count)
Node 0: path=[0, 0], price=100.0, expected=100.0, ✓
Node 1: path=[1, 0], price=100.027401, expected=100.027401, ✓
Node 2: path=[0, 1], price=99.972606, expected=99.972606, ✓
Node 3: path=[2, 0], price=100.05481, expected=100.05481, ✓
Node 4: path=[1, 1], price=100.0, expected=100.0, ✓
Node 5: path=[0, 2], price=99.94522, expected=99.94522, ✓

All pricing formula tests passed! ✓
All structure tests passed! ✓
Test Summary:                 | Pass  Total  Time
Recombining Tree Verification |   25     25  0.0s
Node 0: path=[0, 0], price=100.0, expected=100.0, ✓
Node 1: path=[1, 0], price=100.027401, expected=100.027401, ✓
Node 2: path=[0, 1], price=99.972606, expected=99.972606, ✓
Node 3: path=[2, 0], price=100.05481, expected=100.05481, ✓
Node 4: path=[1, 1], price=100.0, expected=100.0, ✓
Node 5: path=[0, 2], price=99.94522, expected=99.94522, ✓

All pricing formula tests passed! ✓
All structure tests passed! ✓
Test Sum

### Memory Efficiency Demonstration

Compare the memory efficiency of our recombining tree vs a full binary tree. 

> **What this test demonstrates:** We calculate how many nodes a full binary tree would require at our chosen height, then compare this against the actual number of nodes in our recombining implementation. The test also validates that our implementation matches the theoretical predictions from the combinatorial formulas we derived earlier.

The memory comparison becomes increasingly dramatic as tree height increases. For our small example with height `2`, we see a modest reduction, but scaling to realistic financial modeling scenarios with 20+ time steps would show the exponential versus polynomial growth difference that makes recombining trees computationally feasible for real applications.

In [28]:
@testset "Memory Efficiency Tests" begin
    # Compare memory efficiency
    full_tree_nodes = 2^(h+1) - 1  # Full binary tree: 2^0 + 2^1 + ... + 2^h
    recombining_nodes = length(my_tree_model.data)
    theoretical_nodes = Int((h+1)*(h+2)/2)  # Theoretical prediction for binary recombining tree
    
    # Test that recombining tree has fewer nodes than full tree
    @test recombining_nodes < full_tree_nodes
    
    # Test that our implementation matches theoretical prediction
    @test recombining_nodes == theoretical_nodes
    
    # Test that we achieve significant memory reduction for this height
    memory_reduction = (1 - recombining_nodes/full_tree_nodes) * 100
    @test memory_reduction > 0
    
    println("Memory comparison for height h=$h:")
    println("  Full binary tree nodes: $full_tree_nodes")
    println("  Recombining tree nodes: $recombining_nodes") 
    println("  Memory reduction: $(round(memory_reduction, digits=1))%")
    println("  Theoretical prediction: $theoretical_nodes nodes (matches: $(recombining_nodes == theoretical_nodes))")
    println("All memory efficiency tests passed! ✓")
end;

Memory comparison for height h=2:
  Full binary tree nodes: 7
  Recombining tree nodes: 6
  Memory reduction: 14.3%
  Theoretical prediction: 6 nodes (matches: true)
All memory efficiency tests passed! ✓
Test Summary:           | Pass  Total  Time
Memory Efficiency Tests |    3      3  0.0s
  Full binary tree nodes: 7
  Recombining tree nodes: 6
  Memory reduction: 14.3%
  Theoretical prediction: 6 nodes (matches: true)
All memory efficiency tests passed! ✓
Test Summary:           | Pass  Total  Time
Memory Efficiency Tests |    3      3  0.0s


## Summary

We built a recombining price tree in two steps: first we built the structure (connectivity), then we populated it with prices and paths. This separation made it clear what the topology was and what the data were.

Recombination meant that different paths that landed in the same state shared a node, so the tree stayed compact without losing information. You could spot recombination when multiple parents pointed to the same child index in the connectivity map.

Connectivity told us who pointed to whom, and leaves didn't show up as keys because they had no children. Indices were compact labels assigned by the builder.

The data payload was small on purpose: price and a path vector that counted moves aligned with the $\Delta$-vector. There was one record per unique node even if it was reachable by several paths.

Level was derived on the fly as sum(path), which kept the payload light while still giving us everything we needed to traverse, compute time steps, and analyze outcomes.